# OpenAI API

- Use Case: Provides access to advanced AI models developed by OpenAI, including language models (like gpt-5-nano), image generation models, and more. These models can be used for a variety of applications, such as natural language processing, content generation, and complex data analysis.
- API Key: Required for access.
- Pricing: The API usage is metered, and different pricing tiers are available depending on the volume of requests and the specific models used. https://openai.com/pricing
- Website: https://platform.openai.com/docs/overview

**First, you need to get an API key from OpenAI.**

On Colab Notebook, you can add your API key as a secret "OPENAI_API_KEY".

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')
#print(api_key)

In [ ]:
from openai import OpenAI

# If you have set OPENAI_API_KEY properly
client = OpenAI(api_key=api_key)

In [ ]:
# Install the OpenAI package if you haven't yet
#pip install --upgrade openai

In [ ]:
prompt = "Compose a poem that explains the beauty of large language models."

response = client.chat.completions.create(
  model="gpt-5-nano", # Use can choose other models as well, e.g., gpt-5-mini, gpt-40-mini
  messages=[
    {"role": "system", "content": "You are a poetic assistant, skilled in explaining complex programming concepts with creative flair."},
    {"role": "user", "content": prompt}
  ]
)

print(response.choices[0].message.content)

In [ ]:
# gpt-5-nano is one of OpenAI's most cost-efficient models

def get_completion(prompt, model="gpt-5-nano"):
    messages = [
        {"role": "system", "content": "You are an intelligent and articulate assistant."},
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    return response.choices[0].message.content

### Example 1: Use the API to get suggestions on how to analyze a specific dataset.

This example will demonstrate how the API can be used as a tool to brainstorm ideas for data analysis, potentially providing suggestions on types of analysis like time series, product performance, and visualization techniques.

In [ ]:
prompt = """
I have a CSV file containing sales data over the past year. \
The columns include 'Date', 'Product', 'Quantity Sold', and 'Sale Price'. \
I want to analyze this data to gain useful business insights. \
What are some analyses and visualizations I can perform on this data?
"""
print(f"Prompt: {prompt}")

response = get_completion(prompt)
print(f"Response: \n{response}")

### Example 2: Generating Python code

In this example, the API will help translate a data analysis requirement into Python code.

In [ ]:
prompt = """
Write Python script to find the top 3 best-selling products based on the total quantity sold. \
Assume the sales dataframe is named 'sales_data' with columns 'product_id', 'product_name', 'quantity_sold', and 'sale_date'.
"""

print(f"Prompt: {prompt}")

response = get_completion(prompt)
print(f"Response: \n{response}")

### Example 3: Creative Writing Assistance

This example will showcase the AI's ability in creative applications, like enhancing a simple text description. It's a fun way to demonstrate the more artistic and less analytical capabilities of AI.

In [ ]:
prompt = """
I have written a brief description of a beach scene: \
'The beach was quiet. The water was blue. There were a few people around.' \
Can you rewrite this to be more vivid and descriptive?
"""

print(f"Prompt: {prompt}")

response = get_completion(prompt)
print(f"Response: \n{response}")

## More APIs on OpenAI's Platform

### Text-to-speech
Learn how to turn text into lifelike spoken audio.

In [ ]:
def text_to_speech(quote, voice):
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice, # Different voices to choose from: alloy, echo, fable, onyx, nova, shimmer
        input=quote
    )
    return response

In [ ]:
quote = "Success is not final, failure is not fatal: It is the courage to continue that counts."
quote = "You may say I'm a dreamer, but I'm not the only one."
quote = "Magic Mirror on the wall, who is the fairest one of all?"
#quote = "Now I am become Death, the destroyer of worlds."

In [ ]:
voice = "shimmer" #"onyx"
response = text_to_speech(quote, voice)

#response.stream_to_file("../data/output.mp3")

In [ ]:
from IPython.display import Audio

# Attempt to play the binary audio content directly
try:
    audio_content = response.content
    display(Audio(audio_content, autoplay=True))
except AttributeError as e:
    print(f"An error occurred: {e}")

### Image generation
Learn how to generate or manipulate images with DALL·E.

In [ ]:
def get_image_url(prompt):
    response = client.images.generate(
      model="dall-e-3",
      prompt=prompt,
      size="1024x1024",
      quality="standard",
      n=1,
    )
    return response.data[0].url

In [ ]:
from IPython.display import display, Markdown

#prompt="A white siamese cat"
prompt="An ancient castle on a misty cliff, cinematic lighting."
prompt="A close-up of a female Asian student in a university campus, photorealistic detail."
prompt="A bustling New York street at night in anime style, bright neon lights."
prompt="A still life of fruits and wine in the style of a Renaissance painting."
#prompt="A colorful and whimsical illustration of a forest with talking animals, for a children's book."
#prompt="Photorealistic closeup video of two pirate ships battling each other as they sail inside a cup of coffee."

image_url = get_image_url(prompt)
display(Markdown(f"![{prompt}]({image_url})"))

## Vision
The vision models can take in images and answer questions about them.

In [ ]:
import base64

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
from IPython.display import Image

image_path = "https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/img/toilet.jpg"

Image(image_path)

In [ ]:
if image_path.startswith("http://") or image_path.startswith("https://"):
    # If it's a URL, use it directly in the API call
    image_api_payload = {
        "type": "image_url",
        "image_url": {
            "url": image_path
        },
    }
else:
    # If it's a local file, encode it to base64
    base64_image = encode_image(image_path)
    image_api_payload = {
        "type": "image_url",
        "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
        },
    }

response = client.chat.completions.create(
  model="gpt-5-nano",
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "What is happening in this image?",
        },
        image_api_payload,
      ],
    }
  ],
)

print(response.choices[0].message.content)